### Import statements

In [1]:
from speech_signal import run_training, plot_training, rolling_samples, plot_rolling, base_feature_set, all_prefs
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt

### Initialize state

In [2]:
state = {}

### Define window view controls

In [3]:
# work dropdown
work_dd = widgets.Dropdown(
    description = "Work",
    options = [work for work in all_prefs],
)

# prefix dropdown
pref_dd = widgets.Dropdown(
    description = "Book",
    options = [pref for pref in all_prefs[work_dd.value]],
)

# figure viewport
roll_view = widgets.Output()

# handler for work change
def on_work_change(change):
    '''on change to work, update pref options'''
    pref_dd.options = [pref for pref in all_prefs[work_dd.value]]

    # trigger pref update
    on_pref_change(None)

# handler for any change to view
def on_pref_change(change):
    '''redraw the rolling window plot'''
    
    if "test" not in state:
        return
    with roll_view:
        clear_output(wait=True)
        fig = plot_rolling(state["test"], work=work_dd.value, pref=pref_dd.value)
        display(fig)
        plt.close(fig)

# register handlers with widgets
work_dd.observe(on_work_change, names="value")
pref_dd.observe(on_pref_change, names="value")

### Define training controls

In [4]:
# window size
sample_size_input = widgets.BoundedIntText(
    value = 1000,
    min = 200,
    max = 5000,
    description = "Sample size"
)

# random number seed
seed_input = widgets.BoundedIntText(
    value = 0,
    min = 0,
    max = 2**53 - 1,
    description = "Seed"
)

# run button
train_btn = widgets.Button(
    description = "Run training",
)

# model viewport
train_view = widgets.Output()

# handler for button click
def on_train_click(btn):
    '''train model, project rolling samples'''

    # for now just use base features
    feature_set = base_feature_set

    # run training
    state["train"] = run_training(
        feature_set = feature_set,
        sample_size = sample_size_input.value,
        seed = seed_input.value,
    )

    # display model
    with train_view:
        clear_output(wait=True)
        fig = plot_training(state["train"])
        display(fig)
        plt.close(fig)

    # calculate rolling window and project
    state["test"] = rolling_samples(state["train"])

    # trigger rolling viewport update
    on_pref_change(None)

# register handler for train button
train_btn.on_click(on_train_click)

### Start interactive UI

In [5]:
display(widgets.HBox([sample_size_input, seed_input, train_btn]))
display(train_view)
display(widgets.HBox([work_dd, pref_dd]))
display(roll_view)

Output()

Output()